**Part B** - **Classification models on the preprocessed split.**


In [9]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_curve, roc_auc_score,
)

if not os.path.exists('outputs'):
    os.makedirs('outputs')

In [10]:
RANDOM_STATE = 42

X_train = pd.read_csv("/content/X_train.csv")
X_test = pd.read_csv("/content/X_test.csv")

y_train = pd.read_csv("/content/y_train.csv").iloc[:, 0]
y_test = pd.read_csv("/content/y_test.csv").iloc[:, 0]

In [11]:
# ---- Train models -----------------------------------------------------
log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
log_reg.fit(X_train, y_train)

dtree = DecisionTreeClassifier(random_state=RANDOM_STATE)
dtree.fit(X_train, y_train)

models = {"Logistic Regression": log_reg, "Decision Tree": dtree}

In [12]:
# ---- Evaluate -----------------------------------------------------------
metrics_rows = []
roc_data = {}
preds = {}

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    preds[name] = (y_pred, y_proba)

    cm = confusion_matrix(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_proba)
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_data[name] = (fpr, tpr, auc)

    metrics_rows.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1": round(f1, 4),
        "ROC_AUC": round(auc, 4),
        "TN": int(cm[0, 0]), "FP": int(cm[0, 1]),
        "FN": int(cm[1, 0]), "TP": int(cm[1, 1]),
    })

metrics_df = pd.DataFrame(metrics_rows)

In [13]:
# ---- ROC plot -------------------------------------------------------

plt.figure(figsize=(6, 5))
for name, (fpr, tpr, auc) in roc_data.items():
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression vs Decision Tree")
plt.legend()
plt.tight_layout()
plt.savefig("outputs/roc_curves.png", dpi=150)
plt.close()

In [14]:
# ---- Confusion matrix plots -------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, model) in zip(axes, models.items()):
    y_pred = preds[name][0]
    cm = confusion_matrix(y_test, y_pred)
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["No Default", "Default"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["No Default", "Default"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout()
plt.savefig("outputs/confusion_matrices.png", dpi=150)
plt.close()

In [15]:
# ---- Risk-based pricing table (uses logistic regression probabilities) --
log_reg_proba = preds["Logistic Regression"][1]
pricing_df = pd.DataFrame({
    "applicant_index": X_test.index,
    "pred_default_prob": log_reg_proba,
    "actual_default": y_test.values,
})

In [16]:
# Quartile-based tiers, lower risk -> lower rate
pricing_df["risk_tier"] = pd.qcut(
    pricing_df["pred_default_prob"], q=4,
    labels=["Tier 1 (Lowest risk)", "Tier 2 (Low-Med risk)",
            "Tier 3 (Med-High risk)", "Tier 4 (Highest risk)"]
)

rate_ranges = {
    "Tier 1 (Lowest risk)": "9% - 12%",
    "Tier 2 (Low-Med risk)": "13% - 17%",
    "Tier 3 (Med-High risk)": "18% - 24%",
    "Tier 4 (Highest risk)": "25% - 32%",
}
pricing_df["illustrative_rate_range"] = pricing_df["risk_tier"].map(rate_ranges)

pricing_summary = (
    pricing_df.groupby("risk_tier", observed=True)
    .agg(
        n_applicants=("actual_default", "size"),
        avg_predicted_prob=("pred_default_prob", "mean"),
        observed_default_rate=("actual_default", "mean"),
    )
    .reset_index()
)
pricing_summary["illustrative_rate_range"] = pricing_summary["risk_tier"].map(rate_ranges)
pricing_summary["avg_predicted_prob"] = pricing_summary["avg_predicted_prob"].round(4)
pricing_summary["observed_default_rate"] = pricing_summary["observed_default_rate"].round(4)

is_monotonic = pricing_summary["observed_default_rate"].is_monotonic_increasing

In [17]:
# ---- Save everything ----------------------------------------------------
metrics_df.to_csv("outputs/part_b_metrics.csv", index=False)
pricing_summary.to_csv("outputs/risk_pricing_table.csv", index=False)

print("=== Comparison table ===")
print(metrics_df.to_string(index=False))
print("\n=== Risk-based pricing table ===")
print(pricing_summary.to_string(index=False))
print(f"\nMonotonic (lowest->highest risk tier default rate strictly increasing)? {is_monotonic}")

with open("outputs/part_b_report.json", "w") as f:
    json.dump({
        "metrics": metrics_rows,
        "pricing_table": pricing_summary.to_dict(orient="records"),
        "pricing_monotonic": bool(is_monotonic),
    }, f, indent=2)

=== Comparison table ===
              Model  Accuracy  Precision  Recall     F1  ROC_AUC  TN  FP  FN  TP
Logistic Regression      0.76     0.3889    0.35 0.3684   0.7188  69  11  13   7
      Decision Tree      0.65     0.2222    0.30 0.2553   0.5188  59  21  14   6

=== Risk-based pricing table ===
             risk_tier  n_applicants  avg_predicted_prob  observed_default_rate illustrative_rate_range
  Tier 1 (Lowest risk)            25              0.0201                   0.08                9% - 12%
 Tier 2 (Low-Med risk)            25              0.0730                   0.12               13% - 17%
Tier 3 (Med-High risk)            25              0.2341                   0.20               18% - 24%
 Tier 4 (Highest risk)            25              0.5870                   0.40               25% - 32%

Monotonic (lowest->highest risk tier default rate strictly increasing)? True
